<p style="text-align:center"> 
    <a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/" target="_blank"> 
    <img src="../assets/logo.png" width="200" alt="Flavio Aguirre Logo"> 
    </a>
</p>

<h1 align="center"><font size="7"><strong>📉 ByeBye Predictor</strong></font></h1>
<br>    
<br>                            

----

## Data Curation & Preprocessing

**Date**: September 6, 2025<br>
**Subject**: Data preprocessing and cleaning of the Telco Customer Churn dataset."


Based on the conclusive findings from the Exploratory Data Analysis (EDA) notebook, this lab focuses on a single mission: curating the Telco Customer Churn dataset. The objective is to apply a set of systematic transformations to cleanse, normalize, and encode the data, leaving it in an optimal state for the Feature Engineering phase.

No manual feature engineering will be performed in this step. The goal is to create a pristine and consistent database.



### From EDA Findings to Preprocessing Strategy

The EDA provided us with a detailed diagnosis. Now, we translate that diagnosis into a treatment plan. Our tool, `DataProcessor`, will be configured to execute this plan efficiently.

| EDA Finding | Preprocessing Decision | Justification |
| :--- | :--- | :--- |
| **Missing Values ​​Detected** (`plot_missing_values`) | Imputation of numerical values. | Ensures that the dataset does not have gaps that could cause errors in model training. |
| **Outliers Identified** (IQR Method) | Use of robust outlier scaling. | Standardization (`StandardScaler`) centers the data and is a critical first step. Robustness will be provided by imputation with the **median**. |
| **Skewness and Non-Normality Tests** | Standardization of numerical features. | Helps algorithms that assume a normal (or similar) distribution converge faster and perform better. |
| Mixed Data Types (``data_types_overview``) | Clear separation of numerical and categorical columns. | Allows the correct transformation to be applied to each data type (scaling to numerical, encoding to categorical). |
| Categorical Feature Distributions | One-Hot Encoding | Converts categories into a numerical format that the model can understand without assuming a non-existent order between them.



#### Expected final result of Notebook 03
The objective is to apply a set of systematic transformations to cleanse, normalize, and encode the data, leaving it in an optimal state for the Feature Engineering phase.

No manual feature engineering will be performed in this step. The goal is to create a pristine and consistent database.

### We import the necessary libraries

In [1]:
from notebooks_setup import PROJECT_ROOT

from src.utils import get_logger
from src.data_loader import load_csv, preview_df
from src.eda import dataframe_overview
from src.preprocess import DataProcessor

import pandas as pd
import numpy as np

2025-09-06 14:35:34,575 | src.utils | INFO | Added project root to sys.path: C:\Users\Pc\Desktop\github
2025-09-06 14:35:36,108 | src.preprocess | INFO | NLTK resource 'punkt' is already downloaded.
2025-09-06 14:35:36,110 | src.preprocess | INFO | NLTK resource 'stopwords' is already downloaded.
2025-09-06 14:35:36,114 | src.preprocess | WARNING | NLTK resource 'wordnet' not found. Downloading...
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Pc\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
2025-09-06 14:35:36,211 | src.preprocess | INFO | NLTK resource 'punkt_tab' is already downloaded.
2025-09-06 14:35:36,212 | src.preprocess | INFO | Preprocess module loaded correctly and ready to be used.


### Configuring and Executing the `DataProcessor`

With the strategy defined, we instantiate and execute our pipeline. The `DataProcessor` class allows us to encapsulate all this logic in a few lines, ensuring reproducibility.

**Configuration Reasoning:**

* **`imputation_strategy='median'`**: Since the EDA detected outliers (just above the IQR limit, since, as we saw when using the function to plot a box plot, we observed that there are some points above the whiskers), the **median** is a safer and more robust imputation strategy than the mean, as it is not affected by extreme values.
* **`scaling_strategy='standard'`**: The `StandardScaler` is the industry standard and a requirement for many models. It transforms features to have a mean of 0 and a standard deviation of 1.

##### Load dataset and logger

In [2]:
logger = get_logger(__name__)

In [3]:
path_file = "data/interim/telco_customer_churn_snapshot_2025-09-03.csv"
df_telco = load_csv(path_file)
preview_df(df_telco)

2025-09-06 14:35:42,958 | src.data_loader | INFO | Loaded CSV: data/interim/telco_customer_churn_snapshot_2025-09-03.csv | Shape: (7043, 21)
2025-09-06 14:35:42,960 | src.data_loader | INFO | Completed: load_csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# The EDA told us that TotalCharges has blanks that should be null
df_telco['TotalCharges'] = pd.to_numeric(df_telco['TotalCharges'], errors='coerce')
logger.info("Converted column 'TotalCharges' to numeric (dtype=float).")

2025-09-06 14:35:46,358 | __main__ | INFO | Converted column 'TotalCharges' to numeric (dtype=float).


In [5]:
# We instantiate the preprocessor with our evidence-based configuration
preprocessor = DataProcessor(
    imputation_strategy='median',  # Decision informed by the presence of outliers
    scaling_strategy='standard'    # Standard for preparing data for modeling
)

2025-09-06 14:35:48,747 | src.preprocess | INFO | Initializing DataProcessor Class...
2025-09-06 14:35:48,750 | src.preprocess | INFO | DataProcessor initialized successfully!


In [ ]:
# We separate features (X) and target (y)
X = df_telco.drop('Churn', axis=1)
y = df_telco['Churn']

# We run the data curation pipeline
logger.info("Applying preprocessing pipeline (imputation, scaling and encoding)...")
X_processed = preprocessor.process(X)
logger.info("Pipeline completed.")  


2025-09-06 14:35:50,956 | __main__ | INFO | Applying preprocessing pipeline (imputation, scaling and encoding)...
2025-09-06 14:35:50,958 | src.preprocess | INFO | Starting data processing workflow.
2025-09-06 14:35:51,015 | src.preprocess | INFO | Detected comment-like columns: []
2025-09-06 14:35:51,016 | src.preprocess | INFO | Detected high-cardinality noise columns: ['customer_id']
2025-09-06 14:35:51,018 | src.preprocess | INFO | Preprocessing pipeline built successfully.
2025-09-06 14:35:51,019 | src.preprocess | INFO | Fitting and transforming data with the pipeline.
2025-09-06 14:35:51,029 | src.preprocess | INFO | Outlier boundaries calculated for 4 numeric columns.
2025-09-06 14:35:51,065 | src.preprocess | INFO | Data processing complete. Final shape: (7043, 30)
2025-09-06 14:35:51,067 | src.data_loader | INFO | Completed: process
2025-09-06 14:35:51,069 | __main__ | INFO | Pipeline completed.


#### Output Validation and Persistence
A crucial step is to verify that the output is correct and save both the curated data and the adjusted pipeline. The saved pipeline is essential for processing new data in the future using the exact same rules.

In [7]:
# We validate the output
logger.info(f"Dimensions of the processed dataset: {X_processed.shape}")
display(dataframe_overview(X_processed))

2025-09-06 14:35:53,851 | __main__ | INFO | Dimensions of the processed dataset: (7043, 30)
2025-09-06 14:35:53,853 | src.eda | INFO | Generating DataFrame overview.
2025-09-06 14:35:53,864 | src.data_loader | INFO | Completed: dataframe_overview


,dtype,n_unique,n_missing,%_missing
monthly_charges,float64,1585,0,0.0
tenure,float64,73,0,0.0
total_charges,float64,6531,0,0.0
senior_citizen,float64,1,0,0.0
gender_male,float64,2,0,0.0
partner_yes,float64,2,0,0.0
dependents_yes,float64,2,0,0.0
phone_service_yes,float64,2,0,0.0
multiple_lines_no_phone_service,float64,2,0,0.0
multiple_lines_yes,float64,2,0,0.0


We verify that the new dataset does not actually have missing or null values.

In [8]:
assert X_processed.isnull().sum().sum() == 0, logger.warning("Warning: Null values still exist after preprocessing.")
logger.info("Validation successful: No null values were found in the final dataset.")

2025-09-06 14:35:58,285 | __main__ | INFO | Validation successful: No null values were found in the final dataset.


In [9]:
# We save the cured data and the preprocessor
X_processed.to_csv('./data/processed/telco_churn_curated.csv', index=False)
preprocessor.save('./models/preprocessors/telco_preprocessor.joblib')

2025-09-06 14:36:03,471 | src.preprocess | INFO | Preprocessor saved successfully at ./models/preprocessors/telco_preprocessor.joblib.
2025-09-06 14:36:03,473 | src.data_loader | INFO | Completed: save


In [10]:
# Save the target for future use
# It's said to be good practice to save the target aligned with the processed data
np.save('./data/processed/target.npy', y.values)

<br>

---

#### Conclusion

The data curation and preprocessing phase has laid a foundation for the project. All transformations are transparent, reproducible, and align with data science best practices. The curated dataset is now ready for advanced feature engineering and model development.


<br>

<br>
<br>

---

## Author

<a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/">**Flavio Aguirre**</a>
<br>
<a href="https://coursera.org/share/e27ae5af81b56f99a2aa85289b7cdd04">***Data Scientist***</a>